# Modelagem e Simulação - Atividade 04
## Pedro Souza Ferreira & Leonardo Henrique Carvalho

### Problema 1
*Um call center recebe ligações seguindo um processo de Poisson com
taxa de 12 ligações por hora. O tempo que o atendente leva para resolver
um problema é exponencialmente distribuído, com média de 4 minutos.
Modele o número de ligações na fila (incluindo a em atendimento).*

In [ ]:
import numpy as np

def mm1(num_clientes=500000):
    taxa_chegada = 12/60  #~0.2 ligaçoes por minuto
    tempo_medio_atendimento = 4 #4 minutos
    np.random.seed(42)
    inter_chegadas = np.random.exponential(1/taxa_chegada,num_clientes)
    tempos_servico = np.random.exponential(tempo_medio_atendimento,num_clientes)
    tempos_chegada = np.cumsum(inter_chegadas)
    #arrays q guardam os eventos da simulaçao
    tempos_inicio_servico = np.zeros(num_clientes)
    tempos_partida= np.zeros(num_clientes)
    tempos_no_sistema = np.zeros(num_clientes)

    tempos_inicio_servico[0] = tempos_chegada[0]
    tempos_partida[0] =tempos_inicio_servico[0]+ tempos_servico[0]
    tempos_no_sistema[0] = tempos_partida[0] - tempos_chegada[0]

    for i in range(1, num_clientes):
        tempos_inicio_servico[i] = max(tempos_chegada[i], tempos_partida[i-1])
        tempos_partida[i] = tempos_inicio_servico[i] + tempos_servico[i]
        tempos_no_sistema[i] = tempos_partida[i] - tempos_chegada[i]

    burn_in = int(0.1 * num_clientes)

    clientes_validos_sistema = tempos_no_sistema[burn_in:]
    tempo_total_estacionario = tempos_partida[-1]-tempos_chegada[burn_in]
    W_simulado = np.mean(clientes_validos_sistema)
    L_simulado = np.sum(clientes_validos_sistema)/tempo_total_estacionario
    return L_simulado, W_simulado

L, W = mm1(num_clientes=500000)
print(f"(b) numero medio de clientes no sistema(L): {L:.3f} clientes")
print(f"(c) tempo medio gasto no sistema (W): {W:.3f} minutos")

(b) numero medio de clientes no sistema(L): 4.010 clientes
(c) tempo medio gasto no sistema (W): 20.062 minutos


### Problema 2
*Compare o tempo de espera (W) de duas filas M/M/1 independentes
com taxa de chegada λ = 1 cada uma com o de uma fila única M/M/2
com taxa de chegada λ = 2. Calcule o tempo médio que um atendente
fica ocioso em cada caso. Assuma a mesma taxa de serviço para todos
os atendentes.*


In [ ]:
import numpy as np

def MMc(lamb, mu, c, nroClientes):

    deltaChegadas = np.random.exponential(1/lamb, nroClientes)
    chegadas = np.cumsum(deltaChegadas)
    servicos = np.random.exponential(1/mu, nroClientes)

    atendenteLivre = np.zeros(c)
    fimServico = np.zeros(nroClientes)

    for i in range(nroClientes):
        idxAtendente = np.argmin(atendenteLivre)
        inicio = max(chegadas[i], atendenteLivre[idxAtendente])
        fimServico[i] = inicio + servicos[i]
        atendenteLivre[idxAtendente] = fimServico[i]


    tempoSistema = fimServico - chegadas
    tempoTotal = np.max(fimServico)


    ocupacao = np.sum(servicos)/(c * tempoTotal)
    ociosidade = 1 - ocupacao
    return np.mean(tempoSistema), ociosidade



mm11, ocioso_mm11 = MMc(lamb=1.0, mu=1.5, c=1, nroClientes=100000)
mm12, ocioso_mm12 = MMc(lamb=1.0, mu=1.5, c=1, nroClientes=100000)
mm2, ocioso_mm2 = MMc(lamb=2.0, mu=1.5, c=2, nroClientes=100000)

print(f"Resultados com 100000 clientes:")
print(f"M/M/1| tempo de espera(W) medio = {mm11:.4f} | ociosidade = {ocioso_mm11:.2%}")
print(f"M/M/1| tempo de espera(W) medio = {mm12:.4f} | ociosidade = {ocioso_mm12:.2%}")
print(f"M/M/2| tempo de espera(W) medio = {mm2:.4f}  | ociosidade = {ocioso_mm2:.2%}")

Resultados com 100000 clientes:
M/M/1| tempo de espera(W) medio = 1.9677 | ociosidade = 33.67%
M/M/1| tempo de espera(W) medio = 1.9542 | ociosidade = 33.60%
M/M/2| tempo de espera(W) medio = 1.1897  | ociosidade = 33.00%
